In [1]:
# Bootstrap: make the project root importable no matter where this notebook lives.
import os, sys
_here = os.path.abspath(os.getcwd())
_root = _here if os.path.isdir(os.path.join(_here, 'pipeline')) else os.path.abspath(os.path.join(_here, os.pardir))
if _root not in sys.path:
    sys.path.insert(0, _root)
os.chdir(_root)
print('project root:', _root)

project root: /home/temari/god please no diploma/restore_punct


## Run config

In [2]:
from pipeline.env import DATA_DIR, MODEL_DIR
from pipeline.config import RunConfig

cfg = RunConfig(
    name          = "02_baseline_crf_transitions",            # CHANGE-ME per run
    architecture  = "bert+crf",                                 # "bert" | "bert+crf" | "bert+lstm"
    loss          = "crf",                                # "ce" | "focal" | "crf"
    train_files   = [f"{DATA_DIR}/train_all.json"],
    val_files     = [f"{DATA_DIR}/val_internal.json"],
    replay_files  = [],                                     # e.g. [{"path": f"{DATA_DIR}/train_all_plus_synth.json", "n": 15000, "seed": 42}]
    init_from     = None,                                   # e.g. f"{MODEL_DIR}/02_synth_focal" to warm-start
    num_epochs    = 3,
    learning_rate = 2e-5,
    tags          = {"stage": "1-baseline-loss-sweep"},

    crf_init_transitions = True,      # warm-start CRF transition matrix from train data (bert+crf only)
    crf_aux_loss         = "none",   # "none" | "ce_weighted" | "focal" — per-token aux loss on CRF emissions
)
print(cfg)

RunConfig(name='02_baseline_crf_transitions', architecture='bert+crf', loss='crf', train_files=['/home/temari/god please no diploma/restore_punct/data/train_all.json'], val_files=['/home/temari/god please no diploma/restore_punct/data/val_internal.json'], replay_files=[], init_from=None, num_epochs=3, learning_rate=2e-05, benchmarks=None, seed=42, crf_init_transitions=True, crf_aux_loss='none', crf_aux_weight=0.2, crf_aux_gamma=2.0, o_mask_prob=0.0, tags={'stage': '1-baseline-loss-sweep'})


## Train

In [3]:
from pipeline.training import train_run
model = train_run(cfg)

[02_baseline_crf_transitions] architecture=bert+crf loss=crf epochs=3 lr=2e-05


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[02_baseline_crf_transitions] computing empirical CRF transitions from ['/home/temari/god please no diploma/restore_punct/data/train_all.json']...
[02_baseline_crf_transitions] CRF transitions warm-started from data


/home/temari/anaconda3/envs/neural/lib/python3.13/site-packages/torchcrf/__init__.py:249: UserWarning: where received a uint8 condition tensor. This behavior is deprecated and will be removed in a future version of PyTorch. Use a boolean condition instead. (Triggered internally at /pytorch/aten/src/ATen/native/TensorCompare.cpp:614.)
  score = torch.where(mask[i].unsqueeze(1), next_score, score)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,24.604534,22.778923,0.960965,0.962819,0.961347,0.962819
2,21.221501,21.391863,0.963640,0.964814,0.963864,0.964814
3,19.298013,21.231335,0.964192,0.965547,0.964542,0.965547


Model saved -> /home/temari/god please no diploma/restore_punct/models/02_baseline_crf_transitions


## Benchmark + save results

In [4]:
from pipeline.evaluation import evaluate_and_save
reports = evaluate_and_save(cfg)

for test_name, rep in reports.items():
    wa = rep.get('weighted avg', {})
    print(f"{test_name:>14s} | F1={wa.get('f1-score', 0):.4f} P={wa.get('precision', 0):.4f} R={wa.get('recall', 0):.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[02_baseline_crf_transitions] evaluating on General_Test (n=569)


Evaluating:   0%|          | 0/143 [00:00<?, ?it/s]

[02_baseline_crf_transitions] evaluating on GERA_Test (n=1259)


Evaluating:   0%|          | 0/315 [00:00<?, ?it/s]

[02_baseline_crf_transitions] evaluating on LORuGEC_Test (n=603)


Evaluating:   0%|          | 0/151 [00:00<?, ?it/s]

Updated /home/temari/god please no diploma/restore_punct/results/models_db.json (entry: 02_baseline_crf_transitions)
Wrote /home/temari/god please no diploma/restore_punct/results/02_baseline_crf_transitions.json
Wrote /home/temari/god please no diploma/restore_punct/results/02_baseline_crf_transitions.xlsx
  General_Test | F1=0.9470 P=0.9457 R=0.9501
     GERA_Test | F1=0.9575 P=0.9584 R=0.9592
  LORuGEC_Test | F1=0.9662 P=0.9665 R=0.9667


## Example run

In [5]:
from pipeline.inference import load_for_inference, restore_punctuation

m, tok = load_for_inference(cfg)
for t in [
    "Мама мыла раму а папа читал газету",
    "Однако мы решили не идти в кино потому что пошел дождь",
    "Он сказал Привет как дела",
    "Я говорю ей Что думаешь А она мне Да ничего отстань уже",
    "После многих страданий А А Пушкин всё-таки написал свои стишки",
]:
    print(f"Input:    {t}")
    print(f"Restored: {restore_punctuation(m, tok, t)}\n")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Input:    Мама мыла раму а папа читал газету
Restored: Мама мыла раму, а папа читал газету.

Input:    Однако мы решили не идти в кино потому что пошел дождь
Restored: Однако мы решили не идти в кино, потому что пошел дождь.

Input:    Он сказал Привет как дела
Restored: Он сказал: " Привет, как дела".

Input:    Я говорю ей Что думаешь А она мне Да ничего отстань уже
Restored: Я говорю ей: " Что думаешь? А она мне Да ничего, отстань уже!

Input:    После многих страданий А А Пушкин всё-таки написал свои стишки
Restored: После многих страданий А. А. Пушкин всё-таки написал свои стишки.



In [6]:
from pipeline.aggregate import rebuild_master_excel
rebuild_master_excel()

Rebuilt master table -> /home/temari/god please no diploma/restore_punct/results/master_summary.xlsx
  BERT runs   : 4
  Yandex runs : 0


'/home/temari/god please no diploma/restore_punct/results/master_summary.xlsx'